In [4]:
import os

dataset_path = "../../../../Downloads/violence2.v1i.yolov8/test"

label_folder = os.path.join(dataset_path, "labels")
image_folder = os.path.join(dataset_path, "images")

mapping = {
    0: 1,  # nonviolence
    1: 2,  # violence
}

counter = 0

for file in os.listdir(label_folder):

    if not file.endswith(".txt"):
        continue

    label_path = os.path.join(label_folder, file)

    # ---- modify labels ----
    with open(label_path, "r") as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        parts = line.split()
        class_id = int(parts[0])
        parts[0] = str(mapping[class_id])
        new_lines.append(" ".join(parts))
           

    with open(label_path, "w") as f:
        f.write("\n".join(new_lines))

    # ---- rename files ----
    old_name = os.path.splitext(file)[0]
    new_name = f"merged_{counter:06d}"

    new_label_path = os.path.join(label_folder, new_name + ".txt")
    os.rename(label_path, new_label_path)

    # rename corresponding image
    for ext in [".jpg", ".png", ".jpeg"]:
        img_path = os.path.join(image_folder, old_name + ext)
        if os.path.exists(img_path):
            new_img_path = os.path.join(image_folder, new_name + ext)
            os.rename(img_path, new_img_path)
            break

    counter += 1

In [ ]:
import os
import shutil
import random
from collections import defaultdict
import cv2

# --------------------------
# Paths
# --------------------------
dataset_path = "../images/train_all.v1i.yolov8/train"
images_folder = os.path.join(dataset_path, "images")
labels_folder = os.path.join(dataset_path, "labels")

# --------------------------
# Step 1: Count images per class
# --------------------------
class_counts = defaultdict(list)

for label_file in os.listdir(labels_folder):
    if not label_file.endswith(".txt"):
        continue
    label_path = os.path.join(labels_folder, label_file)
    with open(label_path, "r") as f:
        lines = f.readlines()
        classes_in_image = set(int(line.split()[0]) for line in lines)
        for cls in classes_in_image:
            class_counts[cls].append(label_file)

# --------------------------
# Step 2: Find max count (target for balancing)
# --------------------------
max_count = max(len(files) for files in class_counts.values())
print("Class counts before balancing:", {k: len(v) for k, v in class_counts.items()})
print("Target images per class:", max_count)

# --------------------------
# Helper: Augment image and update YOLO labels
# --------------------------
def augment_image_and_labels(img, label_path, augment_ops):
    """Apply augmentations and update YOLO bounding boxes accordingly."""
    # Read labels
    boxes = []
    with open(label_path, "r") as f:
        for line in f.readlines():
            parts = line.strip().split()
            class_id = int(parts[0])
            x, y, w, h = map(float, parts[1:])
            boxes.append([class_id, x, y, w, h])

    h, w = img.shape[:2]

    # Horizontal flip
    if 'flip' in augment_ops and random.random() > 0.5:
        img = cv2.flip(img, 1)
        for b in boxes:
            b[1] = 1 - b[1]  # x_center flipped

    # Rotation
    if 'rotate' in augment_ops and random.random() > 0.5:
        angle = random.choice([90, 180, 270])
        if angle == 90:
            img = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
            for b in boxes:
                b[1], b[2] = b[2], 1 - b[1]
                b[3], b[4] = b[4], b[3]
        elif angle == 180:
            img = cv2.rotate(img, cv2.ROTATE_180)
            for b in boxes:
                b[1], b[2] = 1 - b[1], 1 - b[2]
        elif angle == 270:
            img = cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE)
            for b in boxes:
                b[1], b[2] = 1 - b[2], b[1]
                b[3], b[4] = b[4], b[3]

    # Brightness/contrast
    if 'brightness' in augment_ops:
        alpha = 0.7 + 0.6 * random.random()  # contrast
        beta = random.randint(-30, 30)       # brightness
        img = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)

    return img, boxes

# --------------------------
# Step 3: Duplicate and augment rare-class images
# --------------------------
counter = len(os.listdir(images_folder))

for cls, files in class_counts.items():
    needed = max_count - len(files)
    if needed <= 0:
        continue

    print(f"Duplicating and augmenting {needed} images for class {cls}")
    for _ in range(needed):
        original_label = random.choice(files)
        base_name = os.path.splitext(original_label)[0]

        # Find corresponding image
        img_path = None
        for ext in [".jpg", ".png", ".jpeg"]:
            temp_path = os.path.join(images_folder, base_name + ext)
            if os.path.exists(temp_path):
                img_path = temp_path
                break
        if img_path is None:
            continue

        # Read and augment
        img = cv2.imread(img_path)
        img, new_boxes = augment_image_and_labels(
            img, os.path.join(labels_folder, original_label),
            augment_ops=['flip','rotate','brightness']
        )

        # Save new image and updated labels
        new_name = f"balanced_{counter:06d}"
        new_img_path = os.path.join(images_folder, new_name + os.path.splitext(img_path)[1])
        new_label_path = os.path.join(labels_folder, new_name + ".txt")

        cv2.imwrite(new_img_path, img)

        with open(new_label_path, "w") as f:
            for b in new_boxes:
                class_id, x, y, w_box, h_box = b
                f.write(f"{class_id} {x:.6f} {y:.6f} {w_box:.6f} {h_box:.6f}\n")

        counter += 1

print("Class balancing with safe augmentation complete!")